In [ ]:
import sys
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm 
import pandas as pd
import os

!{sys.executable} -m pip uninstall -y clip
!{sys.executable} -m pip install git+https://github.com/openai/CLIP.git
!{sys.executable} -m pip install ftfy regex tqdm pandas

import clip

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-L/14@336px", device=device)
model.eval() 

input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print(f"Model parameters: {np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print(f"Input resolution: {input_resolution}")

the new one

In [ ]:
import pandas as pd
from PIL import Image, ImageFile
import os

ImageFile.LOAD_TRUNCATED_IMAGES = True

aligned_df = pd.read_csv('aligned_paintings.csv')
mapping_df = pd.read_csv('wikiart_art_pieces.csv')

print(f"Loaded {len(aligned_df)} aligned paintings from Neo4j.")

aligned_df['clean_URL'] = aligned_df['URL'].astype(str).apply(lambda x: x.split('!')[0])
mapping_df['clean_img'] = mapping_df['img'].astype(str).apply(lambda x: x.split('!')[0])

mapping_df = mapping_df.drop_duplicates(subset=['clean_img'], keep='first')

merged_df = pd.merge(aligned_df, mapping_df[['clean_img', 'file_name']], 
                     left_on='clean_URL', right_on='clean_img', how='left')

missing_mappings = merged_df['file_name'].isna().sum()

merged_df.head()

In [ ]:
image_embeddings = []
missing_files = []

image_folder = "./worldkg_images"

with torch.no_grad():
    
    for idx, row in tqdm(merged_df.iterrows(), total=len(merged_df), desc="Extracting CLIP Embeddings"):
        
        if pd.isna(row['file_name']):
            missing_files.append((idx, row['URL'], "Missing from CSV mapping"))
            image_embeddings.append(np.zeros(768))
            continue
            
        local_path = os.path.join(image_folder, str(row['file_name']))
        
        try:
            image = Image.open(local_path).convert("RGB")
            
            image_input = preprocess(image).unsqueeze(0).to(device)
            
            image_features = model.encode_image(image_input)
            
            vector = image_features.cpu().numpy().flatten()
            image_embeddings.append(vector)
            
        except Exception as e:
            missing_files.append((idx, local_path, str(e)))
            image_embeddings.append(np.zeros(512))

if missing_files:
    for err in missing_files[:5]:
        print(err)
else:
    print("\nSuccessfully extracted embeddings!")

In [ ]:
image_embeddings_matrix = np.stack(image_embeddings).astype(np.float32)

np.save('image_embeddings.npy', image_embeddings_matrix)

try:
    graph_embeddings_matrix = np.load('graph_embeddings.npy')
    
    assert image_embeddings_matrix.shape[0] == graph_embeddings_matrix.shape[0], \
    f"Mismatch! Images: {image_embeddings_matrix.shape[0]}, Graphs: {graph_embeddings_matrix.shape[0]}"

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import random

image_folder = "./worldkg_images"

def check_sane_mapping(row):
    if pd.isna(row['file_name']): return False
    
    url_base = str(row['clean_URL']).split('/')[-1].replace('.jpg', '').lower()
    file_base = str(row['file_name']).replace('.jpg', '').lower()
    
    return (url_base in file_base) or (file_base in url_base)

merged_df['is_sane_match'] = merged_df.apply(check_sane_mapping, axis=1)
sane_match_count = merged_df['is_sane_match'].sum()
match_percentage = (sane_match_count / len(merged_df)) * 100

print(f"Total Rows: {len(merged_df)}")
print(f"Rows with logical text matches: {sane_match_count} ({match_percentage:.2f}%)")
if match_percentage > 95:
    print("highly accurate!")


sample_df = merged_df.sample(5)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    local_path = os.path.join(image_folder, str(row['file_name']))
    
    try:
        img = Image.open(local_path)
        ax.imshow(img)
        
        title_text = f"Neo4j: {str(row['ImageName'])[:20]}...\nMapped: {str(row['file_name'])[:20]}..."
        ax.set_title(title_text, fontsize=10)
    except Exception as e:
        ax.set_title("Image Missing/Corrupt", fontsize=10)
        
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

sample_to_print = merged_df.sample(10)
for idx, row in sample_to_print.iterrows():
    neo4j_expected = str(row['clean_URL']).split('/')[-1]
    local_mapped = str(row['file_name'])
    
    print(f"URL Name: {neo4j_expected:<45} | Mapped File: {local_mapped}")


failed_matches = merged_df[merged_df['is_sane_match'] == False]

if len(failed_matches) > 0:    
    for idx, row in failed_matches.head(15).iterrows():
        neo4j_expected = str(row['clean_URL']).split('/')[-1]
        local_mapped = str(row['file_name'])
        
else:
    print("Zero failed string matches!")

In [ ]:
import numpy as np

image_embeds = np.load('image_embeddings.npy')
graph_embeds = np.load('graph_embeddings.npy')

print(f"Image Embeddings Shape: {image_embeds.shape}")
print(f"Graph Embeddings Shape: {graph_embeds.shape}")

In [ ]:
import numpy as np

data = np.load('image_embeddings_vitb32.npy')

print(f"Total images: {data.shape[0]}")
print(f"Max value: {np.max(data)}")
print(f"Min value: {np.min(data)}")

zero_vectors = np.sum(np.all(data == 0, axis=1))

In [ ]:
import os
import torch
from PIL import Image

test_row = merged_df.iloc[0]
local_path = os.path.join(image_folder, str(test_row['file_name']))

print(f"Attempting to load image at: {local_path}")

try:
    image = Image.open(local_path).convert("RGB")
    print("Image loaded !")
    
    image_input = preprocess(image).unsqueeze(0).to(device)
    print("Image preprocessed !")
    
    with torch.no_grad():
        image_features = model.encode_image(image_input)
        
    print(f"Vector shape: {image_features.shape}")

except Exception as e:
    print(e)